# DSA Algorithms API

Validate DSA algorithm utilities with sample inputs.

Steps:
- Run graph algorithms on a small graph.
- Exercise trie and segment tree helpers.
- Capture outputs for regression checks.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from app.dsa_algorithms.graph_algorithms import WeightedEdge, bfs, dijkstra
from app.dsa_algorithms.shortest_paths import bellman_ford

summary = {
    'graph': {},
    'trie': {},
    'segment_tree': {},
}

edges = [
    WeightedEdge(0, 1, 1),
    WeightedEdge(1, 2, 2),
    WeightedEdge(0, 2, 4),
    WeightedEdge(2, 3, 1),
]

summary['graph']['bfs'] = bfs(4, edges, start=0)
summary['graph']['dijkstra'] = dijkstra(4, edges, source=0)
summary['graph']['bellman_ford'] = bellman_ford(4, edges, source=0, directed=False)

print('BFS order:', summary['graph']['bfs']['order'])
print('Dijkstra distances:', summary['graph']['dijkstra']['distances'])
print('Bellman-Ford negative cycle:', summary['graph']['bellman_ford']['negative_cycle'])


In [ ]:
from app.dsa_algorithms.trie import Trie
from app.dsa_algorithms.segment_tree import SegmentTreeLazy

# Trie sample
trie = Trie()
for word in ['alpha', 'alpine', 'beta', 'bet']:
    trie.insert(word)

summary['trie'] = {
    'search_alpha': trie.search('alpha'),
    'search_bet': trie.search('bet'),
    'starts_al': trie.starts_with('al'),
    'delete_beta': trie.delete('beta'),
    'search_beta_after_delete': trie.search('beta'),
}
print('Trie summary:', summary['trie'])

# Segment tree sample
values = [1, 2, 3, 4, 5]
seg = SegmentTreeLazy(values)
seg.range_add(1, 3, 2)
summary['segment_tree'] = {
    'range_sum_0_4': seg.range_sum(0, 4),
    'range_sum_1_3': seg.range_sum(1, 3),
}
print('Segment tree sums:', summary['segment_tree'])


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_dsa_algorithms_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
